# Production - Looping All Countries and Scenarios

## Overview
Top-level orchestration notebook that runs production forecasts for all countries and all scenarios. **Estimated runtime: ~5 hours.**

## Orchestration Flow

**This notebook** → calls → [Production - Run All Scenarios for a Country and Update Heavy Crude Production](#notebook-2957744396429559) → calls → [Production Forecast - Single Scenario by Country](#notebook-212233715354892)

### Data Flow

**Data Sources (Read):**
* `workspace.gold.exogenous_variables` - Scenario definitions and event data

**Data Destinations (Write):**
* None directly - orchestrates child notebooks that write to:
  * `workspace.gold.production_forecast_table_monthly` (via worker notebook)
  * `workspace.gold.production_forecast_table_annual` (via mid-level orchestrator)
  * `workspace.gold.production_model_metrics` (via worker notebook)

### Execution Flow
1. **This notebook** loops through all countries
2. For each country, calls **[Production - Run All Scenarios for a Country and Update Heavy Crude Production](#notebook-2957744396429559)** which:
   * Loops through all scenarios for that country
   * Calls **[Production Forecast - Single Scenario by Country](#notebook-212233715354892)** for each scenario
   * Updates annual forecast table
   * Calculates heavy crude production splits
3. Tracks total execution time across all countries

#Setup the notebook and view all scenarios to run

In [0]:
#Define list of all countries to process
countries = [
    'United States', 'Russia','Saudi Arabia'
    'Canada', 'Iraq', 'China'
    , 'Iran', 'United Arab Emirates', 'Brazil' 
    , 'Kuwait', 'Mexico', 'Kazakhstan' 
    , 'Nigeria', 'Norway', 'Angola' 
    , 'Venezuela', 'Qatar', 'Algeria'
]

total_countries = len(countries)

# Initialize total estimated time before the loop
total_estimated_time = 0

# Loop through each country
for country_idx, country in enumerate(countries, start=1):
    
        
    # Get all unique scenarios for the given country from workspace.gold.exogenous_variables (**only for PRODUCTION & Baseline)
    scenario_list_df = spark.sql(f"""
    SELECT DISTINCT description, Event_Direction
    FROM workspace.gold.exogenous_variables
    WHERE country = '{country}' AND (Event_Direction = 'Production' OR Event_Direction = 'Baseline')
    """)
    
    if scenario_list_df.count() == 0:
        print(f"⚠ No scenarios found for {country}. Skipping...")
        continue
    
    # Calculate total scenarios and estimated time
    total_scenarios = scenario_list_df.count()
    estimated_minutes = total_scenarios * 9  # Assuming 9 minutes per scenario

    
    # Accumulate total estimated time
    total_estimated_time += estimated_minutes
    


print(f"\n\n{'='*80}")
print(f"✓ ALL {total_countries} COUNTRIES + SCENARIOS READY TO RUN")
print(f"Total estimated time to run all scenarios is {total_estimated_time} minutes.")
print(f"{'='*80}")

⚠ No scenarios found for Saudi ArabiaCanada. Skipping...


✓ ALL 17 COUNTRIES + SCENARIOS READY TO RUN
Total estimated time to run all scenarios is 306 minutes.


#Run the Production Forecasts

In [0]:
# Loop through all countries and run production forecasts for each
import time

print(f"\n{'='*80}")
print(f"STARTING PRODUCTION FORECASTS FOR ALL COUNTRIES")
print(f"{'='*80}\n")

# Initialize total time tracker before the loop
total_time_taken = 0

for country_idx, country in enumerate(countries, start=1):
    print(f"{'#'*70}")
    print(f"COUNTRY {country_idx}/{total_countries}: {country}")
    print(f"{'#'*70}\n")
    
    # Get all unique Production scenarios for the given country from workspace.gold.exogenous_variables
    scenario_list_df = spark.sql(f"""
    SELECT DISTINCT description
    FROM workspace.gold.exogenous_variables
    WHERE country = '{country}' AND (Event_Direction = 'Production' OR Event_Direction = 'Baseline')
    """)
    
    # Collect all scenarios into a list
    scenarios = [row.description for row in scenario_list_df.collect()]
    total_scenarios = len(scenarios)
    
    if total_scenarios == 0:
        print(f"⚠ No Production scenarios found for {country}. Skipping...")
        continue
    
    print(f"{'='*60}")
    print(f"Starting PRODUCTION forecast loop for {country}")
    print(f"Total scenarios to process: {total_scenarios}")
    print(f"{'='*60}")
    
    # Record start time for this country
    start_time = time.time()

    # Loop through each scenario and run the forecast
    for idx, scenario in enumerate(scenarios, start=1):
        print(f"[{idx}/{total_scenarios}] Processing scenario: {scenario}")
        print("-" * 60)
        
        try:
            dbutils.notebook.run(
                '/Workspace/Users/mijimorgan@gmail.com/Modelling Workflow Scripts/2-Forecasting/Worker Notebooks/Production - Run All Scenarios for a Country and Update Heavy Crude Production',
                3600,  # 60 minute timeout
                {'scenario': scenario, 'country': country}
            )
            print(f"✓ Completed {idx}/{total_scenarios} scenarios ({int(idx/total_scenarios*100)}% done for {country})\n")
        except Exception as e:
            print(f"✗ ERROR running scenario '{scenario}' for {country}: {str(e)}\n")
            continue
    
    # Calculate time for this country
    end_time = time.time()
    country_time = end_time - start_time
    total_time_taken += country_time
    
    minutes = int(country_time // 60)
    seconds = int(country_time % 60)
    
    print(f"\n{'='*60}")
    print(f"✓ COMPLETED {country} ({country_idx}/{total_countries} countries)")
    print(f"Time taken for {country}: {minutes} minutes and {seconds} seconds")
    print(f"Total time elapsed so far: {int(total_time_taken // 60)} minutes and {int(total_time_taken % 60)} seconds")
    print(f"{'='*60}\n\n")

# Calculate and display final execution time
total_minutes = int(total_time_taken // 60)
total_seconds = int(total_time_taken % 60)

print("\n" + "="*80)
print(f"✓ ALL {total_countries} COUNTRIES COMPLETED")
print(f"Total execution time: {total_minutes} minutes and {total_seconds} seconds")
print("="*80)


STARTING PRODUCTION FORECASTS FOR ALL COUNTRIES

######################################################################
COUNTRY 1/1: Angola
######################################################################

Starting PRODUCTION forecast loop for Angola
Total scenarios to process: 1
[1/1] Processing scenario: Baseline scenario
------------------------------------------------------------
